# scRareBench | High-Level Harmony | Dataset 2 — mBDRC renal cortex

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Google Colab notebook runs **Harmony** on scRareBench **Dataset 2 (`mbdrc_renal_cortex`)** through the generic **`MethodSpec → benchmark_method`** API.

Harmony remains user-owned method code. scRareBench loads Dataset 2, runs independent method seeds, validates latent/cell alignment, evaluates each run, and builds the multi-seed deliverable.

For direct comparability with the other Dataset 2 high-level notebooks, the method seeds are:

`42, 123, 2026`

The Harmony-side pipeline follows the audited Dataset 2 policy:

`raw counts → global Seurat-v3 4000 HVGs → normalize_total(10000) → log1p → scale(max=10) → PCA50 → Harmony`

The combined registered Dataset 2 batch (`donor_id × assay`) is passed to **Harmony**, but deliberately **not** to HVG selection. The benchmark label is never used by the integration method.

> **Runtime note:** Harmony is CPU-based. Scaling the selected 4,000 genes can require substantial RAM, so a high-memory Colab runtime is recommended. For a quick smoke test, temporarily use `METHOD_SEEDS = [42]`.

In [ ]:
# Optional diagnostic only: leave commented unless you want to inspect the runtime.
# import sys
# from importlib import metadata
# print(sys.version)
# for pkg in ("numpy", "pandas", "scipy", "anndata", "scanpy", "harmonypy", "scrarebench"):
#     try:
#         print(pkg, metadata.version(pkg))
#     except metadata.PackageNotFoundError:
#         print(pkg, "not installed")


In [ ]:
import subprocess
import sys

REPO = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@cfdb720f5488b9ceef9a225b35fc7558bcb0251c"

# Install the lightweight package first; setup_runtime handles its runtime dependencies.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", REPO
])

# Method dependencies stay explicitly user-controlled.
METHOD_DEPENDENCIES = ("harmonypy==2.0.0",)
INSTALL_METHOD_DEPENDENCIES = True

from scrarebench.runtime import setup_runtime

runtime_report = setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES if INSTALL_METHOD_DEPENDENCIES else (),
    extra_imports=("harmonypy",) if INSTALL_METHOD_DEPENDENCIES else (),
    quiet=False,
)


## Load Dataset 2

The built-in loader returns the registered mBDRC renal-cortex dataset and its scRareBench evaluation metadata. No Harmony-specific normalization or integration is performed by the loader.


In [ ]:
from scrarebench import load_dataset, dataset_info
import numpy as np
import pandas as pd
from scipy import sparse

adata = load_dataset(2)
info = dataset_info(adata)

assert info.get("dataset_index") == 2
assert info.get("batch_key") in adata.obs.columns
assert info.get("scib_hvg_batch_mode") == "global", info
assert adata.obs_names.is_unique


def _sampled_count_diagnostics(matrix, max_values=100_000, seed=42):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix).ravel()
    if len(values) > max_values:
        rng = np.random.default_rng(seed)
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    return {
        "finite_fraction": float(np.mean(np.isfinite(values))) if len(values) else 1.0,
        "nonnegative": bool(len(finite) == 0 or finite.min() >= 0),
        "integer_like": bool(len(finite) == 0 or np.allclose(finite, np.rint(finite), atol=1e-6)),
    }


def _copy_matrix(matrix):
    return matrix.copy().tocsr() if sparse.issparse(matrix) else np.asarray(matrix).copy()


def ensure_counts_layer(adata, layer_key="counts"):
    if layer_key in adata.layers:
        diag = _sampled_count_diagnostics(adata.layers[layer_key])
        if diag["finite_fraction"] == 1.0 and diag["nonnegative"] and diag["integer_like"]:
            return f"layers/{layer_key}", diag
        raise ValueError(f"Existing {layer_key!r} layer is not valid raw counts: {diag}")

    if adata.raw is not None:
        raw_names = pd.Index(adata.raw.var_names.astype(str))
        current_names = pd.Index(adata.var_names.astype(str))
        positions = raw_names.get_indexer(current_names)
        if np.all(positions >= 0):
            matrix = adata.raw.X[:, positions]
            diag = _sampled_count_diagnostics(matrix)
            if diag["finite_fraction"] == 1.0 and diag["nonnegative"] and diag["integer_like"]:
                adata.layers[layer_key] = _copy_matrix(matrix)
                return "raw.X aligned to adata.var_names", diag

    diag = _sampled_count_diagnostics(adata.X)
    if diag["finite_fraction"] == 1.0 and diag["nonnegative"] and diag["integer_like"]:
        adata.layers[layer_key] = _copy_matrix(adata.X)
        return "X", diag

    raise ValueError(
        "Could not locate nonnegative integer-like raw counts for Dataset 2. "
        f"layers={list(adata.layers)}, raw_present={adata.raw is not None}, X_diagnostics={diag}"
    )


count_layer = info.get("count_layer") or "counts"
count_source, count_diag = ensure_counts_layer(adata, count_layer)

print(adata)
print("Dataset:", info.get("name"))
print("Batch key:", info.get("batch_key"))
print("Label key:", info.get("label_key"))
print("HVG batch mode:", info.get("scib_hvg_batch_mode"))
print("Count layer:", count_layer)
print("Count source:", count_source)
print("Count diagnostics:", count_diag)
print("Cells:", f"{adata.n_obs:,}")
print("Features:", f"{adata.n_vars:,}")


## Reproducibility and Harmony configuration

The method seed changes between Harmony runs, while the benchmark seed remains fixed at 42. The parameters below match the audited Dataset 2 Harmony workflow.


In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    # Dataset 2 fixed feature policy
    "n_hvg": 4000,
    "hvg_policy": "global_seurat_v3_raw_counts",
    "hvg_batch_key": None,
    "hvg_span": 0.3,

    # Expression preprocessing
    "normalize_target_sum": 10_000.0,
    "scale_data": True,
    "scale_max_value": 10.0,
    "n_pcs": 50,
    "pca_solver": "arpack",

    # harmonypy 2.0.0 settings
    "theta": 2.0,
    "lamb": None,
    "sigma": 0.1,
    "nclust": None,
    "tau": 0.0,
    "block_size": 0.05,
    "max_iter_harmony": 10,
    "max_iter_kmeans": 4,
    "epsilon_cluster": 1e-3,
    "epsilon_harmony": 1e-2,
    "alpha": 0.2,
    "batch_prop_cutoff": 1e-5,
    "ncores": 0,
    "verbose": True,
}

OUTPUT_DIR = "/content/scrarebench_harmony_dataset2_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_harmony_dataset2_method_runs"

print("Method seeds:", METHOD_SEEDS)
print("Benchmark seed:", BENCHMARK_SEED)


## User-owned Harmony method

This is the only Harmony-specific section.

For each method seed, the runner:

1. keeps RNA/GEX features when modality metadata are present;
2. validates the raw-count layer;
3. selects **4000 global Seurat-v3 HVGs** from raw counts, with no batch key;
4. normalizes, log-transforms and scales expression;
5. computes PCA50;
6. runs Harmony using the registered Dataset 2 integration/evaluation batch;
7. validates cell order and latent finiteness;
8. saves Harmony objective history as provenance when available.

The biological benchmark label is never supplied to Harmony.


In [ ]:
import gc
import time
from pathlib import Path

import harmonypy as hm
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

from scrarebench import MethodOutput, dataset_info


def _gex_mask(method_adata):
    """Return a robust RNA/GEX feature mask without using benchmark labels."""
    for key in ("feature_types", "feature_type", "modality"):
        if key not in method_adata.var.columns:
            continue
        values = method_adata.var[key].astype(str).str.lower()
        mask = values.str.contains(r"gene expression|gex|rna", regex=True, na=False).to_numpy()
        if mask.any():
            return mask
    return np.ones(method_adata.n_vars, dtype=bool)


def _validate_count_like(matrix, *, seed, max_values=100_000):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix).ravel()
    values = np.asarray(values)
    if values.size > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[rng.choice(values.size, size=max_values, replace=False)]
    values = values.astype(float, copy=False)
    if values.size and not np.isfinite(values).all():
        raise ValueError("Raw count layer contains NaN or infinite values.")
    if values.size and np.min(values) < 0:
        raise ValueError("Raw count layer contains negative values.")
    if values.size and not np.allclose(values, np.rint(values), rtol=0.0, atol=1e-6):
        raise ValueError("The declared count layer does not look like integer raw counts.")


def run_harmony(method_adata, seed, config):
    meta = dataset_info(method_adata)
    batch_key = str(meta.get("batch_key") or "")
    count_layer = str(meta.get("count_layer") or "counts")

    if not batch_key or batch_key not in method_adata.obs.columns:
        raise KeyError("Dataset 2 batch metadata is missing.")
    if count_layer not in method_adata.layers:
        raise KeyError(f"Raw count layer {count_layer!r} is required for Harmony preprocessing.")

    seed = int(seed)
    np.random.seed(seed)

    original_barcodes = method_adata.obs_names.astype(str).to_numpy(copy=True)
    if len(np.unique(original_barcodes)) != len(original_barcodes):
        raise ValueError("Dataset 2 observation names must be unique.")

    feature_mask = _gex_mask(method_adata)
    work = method_adata[:, feature_mask].copy()
    work.var_names_make_unique()

    _validate_count_like(work.layers[count_layer], seed=seed)

    # Dataset 2 policy: GLOBAL Seurat-v3 HVGs from raw counts.
    # The integration/evaluation batch is deliberately not passed here.
    sc.pp.highly_variable_genes(
        work,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), work.n_vars),
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )

    hvg_mask = work.var["highly_variable"].fillna(False).to_numpy(dtype=bool)
    expected_hvg = min(int(config["n_hvg"]), work.n_vars)
    observed_hvg = int(hvg_mask.sum())
    if observed_hvg != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} global Dataset 2 HVGs, but selected {observed_hvg}."
        )

    work = work[:, hvg_mask].copy()

    current_barcodes = work.obs_names.astype(str).to_numpy()
    if not np.array_equal(current_barcodes, original_barcodes):
        raise RuntimeError("Cell order changed during Harmony feature preprocessing.")

    counts = work.layers[count_layer]
    if sparse.issparse(counts):
        work.X = counts.tocsr().astype(np.float32).copy()
    else:
        work.X = np.asarray(counts, dtype=np.float32).copy()

    sc.pp.normalize_total(work, target_sum=float(config["normalize_target_sum"]))
    sc.pp.log1p(work)

    if bool(config["scale_data"]):
        sc.pp.scale(work, max_value=float(config["scale_max_value"]))

    n_pcs = min(int(config["n_pcs"]), work.n_vars - 1, work.n_obs - 1)
    if n_pcs < 2:
        raise ValueError(f"Too few dimensions for PCA: {n_pcs}")

    sc.pp.pca(
        work,
        n_comps=n_pcs,
        zero_center=True,
        svd_solver=str(config["pca_solver"]),
        random_state=seed,
    )

    pca = np.asarray(work.obsm["X_pca"], dtype=np.float64)
    if pca.shape != (work.n_obs, n_pcs):
        raise RuntimeError(f"Unexpected PCA shape: {pca.shape}")
    if not np.isfinite(pca).all():
        raise ValueError("PCA representation contains NaN or infinite values.")

    print(
        f"[Harmony seed={seed}] cells={work.n_obs:,}, HVGs={work.n_vars:,}, "
        f"PCs={n_pcs}, batches={work.obs[batch_key].nunique()}"
    )

    started = time.perf_counter()
    harmony_out = hm.run_harmony(
        pca,
        work.obs,
        batch_key,
        theta=config["theta"],
        lamb=config["lamb"],
        sigma=config["sigma"],
        nclust=config["nclust"],
        tau=config["tau"],
        block_size=config["block_size"],
        max_iter_harmony=config["max_iter_harmony"],
        max_iter_kmeans=config["max_iter_kmeans"],
        epsilon_cluster=config["epsilon_cluster"],
        epsilon_harmony=config["epsilon_harmony"],
        alpha=config["alpha"],
        batch_prop_cutoff=config["batch_prop_cutoff"],
        verbose=config["verbose"],
        random_state=seed,
        ncores=config["ncores"],
    )
    elapsed = time.perf_counter() - started

    latent = np.asarray(harmony_out.Z_corr, dtype=np.float32)
    if latent.ndim != 2:
        raise RuntimeError(f"Harmony returned a non-2D latent: {latent.shape}")

    # harmonypy 2.0.0 normally returns cells × PCs; retain a defensive guard.
    if latent.shape[0] != work.n_obs and latent.shape[1] == work.n_obs:
        latent = latent.T

    if latent.shape[0] != work.n_obs or latent.shape[1] < 2:
        raise RuntimeError(
            f"Unexpected Harmony latent shape {latent.shape}; expected {work.n_obs} rows."
        )
    if not np.isfinite(latent).all():
        raise ValueError("Harmony latent contains NaN or infinite values.")

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    provenance = {}

    objective_harmony = np.asarray(
        getattr(harmony_out, "objective_harmony", []), dtype=float
    ).ravel()
    objective_kmeans = np.asarray(
        getattr(harmony_out, "objective_kmeans", []), dtype=float
    ).ravel()

    if len(objective_harmony) or len(objective_kmeans):
        max_len = max(len(objective_harmony), len(objective_kmeans))
        history = pd.DataFrame({
            "iteration": np.arange(max_len, dtype=int),
            "objective_harmony": np.pad(
                objective_harmony,
                (0, max_len - len(objective_harmony)),
                constant_values=np.nan,
            ),
            "objective_kmeans": np.pad(
                objective_kmeans,
                (0, max_len - len(objective_kmeans)),
                constant_values=np.nan,
            ),
        })
        history_path = seed_dir / "harmony_objectives.csv"
        history.to_csv(history_path, index=False)
        provenance["harmony_objectives"] = history_path

    print(
        f"[Harmony seed={seed}] latent={latent.shape}, integration_time={elapsed:.2f}s"
    )

    barcodes = work.obs_names.astype(str).to_numpy()
    del harmony_out, work, pca
    gc.collect()

    return MethodOutput(
        latent=latent,
        barcodes=barcodes,
        representation_key="X_pca_harmony",
        provenance_files=provenance,
    )


## Run the three-seed High-Level benchmark

`benchmark_method` executes the complete Harmony runner independently for each method seed while keeping benchmark-side randomness fixed at 42.

For a first smoke test, temporarily set `METHOD_SEEDS = [42]`; restore all three seeds for the final run.


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="Harmony",
    runner=run_harmony,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "include_latent_in_bundle": False,
    },
    # Dependencies were already handled above by setup_runtime.
    install_dependencies=False,
    finalize=True,
)


## Inspect the multi-seed result


In [ ]:
from IPython.display import display

display(result.summary().round(5))

print("Completed method seeds:", result.seeds)
print("Multi-seed report:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)


## Optional: inspect each seed separately


In [ ]:
for seed in result.seeds:
    run = result.runs[seed]
    print(f"\n===== Harmony seed {seed} =====")
    print("Output directory:", run.output_dir)
    try:
        display(run.summary().round(5))
    except Exception:
        # Keep this diagnostic cell compatible if a single-run result object
        # exposes its tables through attributes rather than summary().
        print(run)


## Download the complete output archive


In [ ]:
from pathlib import Path
from google.colab import files

archive_path = Path(result.archive_path)

if not archive_path.exists():
    raise FileNotFoundError(
        f"Output archive was not found: {archive_path}\n"
        "Make sure benchmark_method(..., finalize=True) completed successfully."
    )

print(f"Downloading: {archive_path.name}")
print(f"Size: {archive_path.stat().st_size / (1024**2):.2f} MB")

files.download(str(archive_path))
